# Real Estate Tracker - Exploração de Fontes de Dados

Notebook exploratório para validar cada fonte de dados disponível para o app de rentabilidade imobiliária.

## Seções
1. BCB SGS API — Selic, IPCA, IGP-M
2. IBGE — Municípios e População
3. Ipeadata — INCC, séries econômicas
4. Simulação de Card — exemplo completo de análise

In [ ]:
import sys
sys.path.insert(0, '../backend')

from datetime import date
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

## 1. BCB SGS API — Índices Econômicos

In [ ]:
from src.data_sources.bcb import BCBClient

bcb = BCBClient()

# Buscar Selic, IPCA e IGP-M desde 2020
inicio = date(2020, 1, 1)

df_selic = bcb.get_selic(data_inicio=inicio)
df_ipca = bcb.get_ipca(data_inicio=inicio)
df_igpm = bcb.get_igpm(data_inicio=inicio)

print(f'Selic: {len(df_selic)} registros, último: {df_selic["valor"].iloc[-1]}')
print(f'IPCA: {len(df_ipca)} registros, último: {df_ipca["valor"].iloc[-1]}')
print(f'IGP-M: {len(df_igpm)} registros, último: {df_igpm["valor"].iloc[-1]}')

In [ ]:
# Gráfico: IPCA vs IGP-M mensal
fig, ax = plt.subplots()
ax.plot(df_ipca['data'], df_ipca['valor'], label='IPCA', linewidth=1.5)
ax.plot(df_igpm['data'], df_igpm['valor'], label='IGP-M', linewidth=1.5, alpha=0.8)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('IPCA vs IGP-M — Variação Mensal (%)')
ax.set_ylabel('Variação mensal (%)')
ax.legend()
plt.tight_layout()
plt.show()

# IPCA acumulado 12 meses
ipca_12m = bcb.calcular_acumulado(df_ipca, 12)
igpm_12m = bcb.calcular_acumulado(df_igpm, 12)
print(f'\nIPCA acumulado 12m: {ipca_12m:.2f}%')
print(f'IGP-M acumulado 12m: {igpm_12m:.2f}%')

## 2. IBGE — Municípios e População

In [ ]:
from src.data_sources.ibge import IBGEClient

ibge = IBGEClient()

# Top 10 municípios mais populosos de SP
df_municipios_sp = ibge.get_municipios('SP')
print(f'Total de municípios em SP: {len(df_municipios_sp)}')
print(df_municipios_sp.head(10))

In [ ]:
# População de São Paulo capital
df_pop_sp = ibge.get_populacao('3550308')
print(f'\nPopulação São Paulo capital (Censo 2022):')
print(df_pop_sp)

## 3. Ipeadata — INCC e Séries Econômicas

In [ ]:
from src.data_sources.ipeadata import IpeadataClient

ipea = IpeadataClient()

# INCC mensal
df_incc = ipea.get_incc()
print(f'INCC-DI mensal: {len(df_incc)} registros')
print(df_incc.tail())

In [ ]:
# Gráfico: INCC últimos 5 anos
df_incc_recente = df_incc[df_incc['data'] >= '2020-01-01']

fig, ax = plt.subplots()
ax.bar(df_incc_recente['data'], df_incc_recente['valor'], width=25, alpha=0.7)
ax.set_title('INCC-DI — Variação Mensal (%) — Últimos 5 anos')
ax.set_ylabel('Variação mensal (%)')
plt.tight_layout()
plt.show()

## 4. Simulação de Card — Exemplo Completo

Simulação de um imóvel fictício em São Paulo para demonstrar todos os cálculos.

In [ ]:
from src.services.yield_calc import YieldService
from src.services.benchmark import BenchmarkService

yield_svc = YieldService()
bench_svc = BenchmarkService()

# Imóvel fictício: Apto 70m² em Pinheiros, SP
IMOVEL = {
    'nome': 'Apto Pinheiros',
    'valor_compra': 750_000,
    'data_compra': date(2022, 6, 1),
    'area_m2': 70,
    'aluguel_mensal': 4_200,
    'iptu_anual': 3_600,
    'condominio_mensal': 900,
    'seguro_anual': 600,
}

print(f"=== {IMOVEL['nome']} ===")
print(f"Valor de compra: R$ {IMOVEL['valor_compra']:,.0f}")
print(f"Área: {IMOVEL['area_m2']} m²")
print(f"Aluguel: R$ {IMOVEL['aluguel_mensal']:,.0f}/mês")
print(f"Preço/m²: R$ {IMOVEL['valor_compra']/IMOVEL['area_m2']:,.0f}")

In [ ]:
# Cálculo de yield
resultado_yield = yield_svc.yield_liquido(
    valor_imovel=IMOVEL['valor_compra'],
    aluguel_mensal=IMOVEL['aluguel_mensal'],
    iptu_anual=IMOVEL['iptu_anual'],
    condominio_mensal=IMOVEL['condominio_mensal'],
    seguro_anual=IMOVEL['seguro_anual'],
    taxa_administracao_pct=8,  # imobiliária
    vacancia_pct=5,  # 5% vacância
)

print('\n--- Yield ---')
for k, v in resultado_yield.items():
    if k != 'breakdown':
        print(f'  {k}: {v}')
print('\n  Breakdown custos:')
for k, v in resultado_yield['breakdown'].items():
    print(f'    {k}: R$ {v:,.0f}')

In [ ]:
# Comparação: e se fosse Airbnb?
resultado_airbnb = yield_svc.yield_airbnb(
    valor_imovel=IMOVEL['valor_compra'],
    diaria_media=350,
    taxa_ocupacao_pct=65,
    custos_fixos_mensal=IMOVEL['condominio_mensal'] + IMOVEL['iptu_anual']/12,
    custos_limpeza_por_estadia=120,
    media_noites_por_estadia=3,
)

print('\n--- Yield Airbnb ---')
for k, v in resultado_airbnb.items():
    if k != 'breakdown':
        print(f'  {k}: {v}')

# Comparação
comp = yield_svc.comparar_longterm_vs_airbnb(resultado_yield, resultado_airbnb)
print(f'\n--- Comparação ---')
print(f'  Long-term: {comp["longterm_yield"]:.2f}% a.a.')
print(f'  Airbnb: {comp["airbnb_yield"]:.2f}% a.a.')
print(f'  Diferença: {comp["diferenca_pp"]:+.2f}pp')
print(f'  Recomendação: {comp["recomendacao"]}')

In [ ]:
# Benchmark vs renda fixa
yield_liquido = resultado_yield['yield_liquido']
comparacao = bench_svc.comparar_com_renda_fixa(yield_liquido)

print(f'\n--- Benchmark vs Renda Fixa ---')
print(f'  Yield imóvel: {comparacao["imovel_yield"]:.2f}%')
print(f'  Yield real: {comparacao["imovel_yield_real"]:.2f}%')
print()
for nome, dados in comparacao['comparacoes'].items():
    spread = dados['spread_pp']
    print(f'  {nome}: spread {spread:+.2f}pp')

In [ ]:
# Custo de oportunidade: imóvel vs CDI em diferentes horizontes
oportunidade = bench_svc.custo_oportunidade(
    valor_imovel=IMOVEL['valor_compra'],
    yield_imovel=yield_liquido,
    valorizacao_anual_pct=5.0,  # estimativa FipeZAP
)

print('\n--- Custo de Oportunidade: Imóvel vs CDI ---')
print(f'  Retorno total imóvel: {oportunidade["retorno_total_imovel_aa"]:.2f}% a.a.')
print(f'  CDI líquido: {oportunidade["cdi_liquido_aa"]:.2f}% a.a.')
print()
for periodo, dados in oportunidade['projecoes'].items():
    print(f'  {periodo}:')
    print(f'    Imóvel: R$ {dados["imovel_valor"]:,.0f} (ganho R$ {dados["imovel_ganho"]:,.0f})')
    print(f'    CDI:    R$ {dados["cdi_valor"]:,.0f} (ganho R$ {dados["cdi_ganho"]:,.0f})')
    print(f'    Melhor: {dados["melhor"]} (diferença R$ {dados["diferenca"]:,.0f})')
    print()